# Construyendo aplicaciones de IA con Gradio y NVIDIA NIM

En los notebooks anteriores interactuamos con LLMs directamente desde Python.
En este notebook construiremos interfaces web interactivas usando **Gradio**,
conectadas a **NVIDIA NIM** y al modelo **Nemotron 3 Super**.

Gradio permite crear interfaces web para modelos de IA con muy poco código. Es
muy útil para demostraciones, prototipos y herramientas internas.

## Prerrequisitos

Asegúrate de tener `NVIDIA_API_KEY` en el archivo `.env`. Si todavía no la
tienes, revisa las instrucciones del notebook 03.

In [ ]:
# %%bash
!pip install -q --upgrade gradio openai python-dotenv

## 1. Configuración del Entorno

In [1]:
import os
import gradio as gr
from openai import OpenAI
from dotenv import load_dotenv

In [2]:
# Cargamos las variables de entorno desde el archivo .env
from google.colab import userdata
if userdata.get('NVIDIA_API_KEY'):
    print("NVIDIA API Key cargada correctamente")
else:
    print("ERROR: NVIDIA_API_KEY no encontrada. Verifica tu archivo .env")


NVIDIA API Key cargada correctamente


In [3]:
# Cliente compatible con OpenAI conectado a NVIDIA NIM
client = OpenAI(
    base_url="https://integrate.api.nvidia.com/v1",
    api_key=userdata.get('NVIDIA_API_KEY'),
)

In [4]:
# Modelo usado en todo el notebook
MODELO = "nvidia/nemotron-3-super-120b-a12b"

## 2. Tu primera interfaz con Gradio

Antes de conectar el LLM, entendamos cómo funciona Gradio con un ejemplo simple.

`gr.Interface` toma una función de Python y la convierte en una interfaz web:

- `fn`: función que se ejecutará.
- `inputs`: componentes de entrada.
- `outputs`: componentes de salida.

In [5]:
# Función simple que devuelve lo que el usuario escribe
def echo(message):
    return f"Dijiste: {message}"

In [6]:
# Creamos la interfaz — Gradio levanta un servidor web local automáticamente
# Al ejecutar esta celda verás un link para abrir la interfaz en el navegador
demo = gr.Interface(
    fn=echo,
    inputs="textbox",
    outputs="textbox",
    title="Echo Bot",
    description="Una interfaz simple que repite lo que escribes",
    flagging_mode="never"  # desactiva el botón de reportar respuestas
)

In [7]:
demo.launch(
    server_name="0.0.0.0",
    server_port=8081,
    show_error=True
)

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://9e0f160801abb7613c.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [8]:
demo.close()

Closing server running on port: 8081


## 3. Chatbot básico con NVIDIA NIM

Ahora conectamos Gradio con NVIDIA NIM. La función `get_response` llama a la API
y devuelve el texto; Gradio se encarga de presentarlo en la interfaz.

In [9]:
def get_response(message: str) -> str:
    """Obtiene una respuesta de NVIDIA NIM."""
    response = client.chat.completions.create(
        model=MODELO,
        messages=[
            {"role": "system", "content": "Eres un asistente útil."},
            {"role": "user", "content": message},
        ],
        temperature=1.0,
        top_p=0.95,
    )
    return response.choices[0].message.content

In [10]:
chatbot = gr.Interface(
    fn=get_response,
    inputs=gr.Textbox(placeholder="Escribe tu pregunta aquí..."),
    outputs="text",
    title="Chatbot con NVIDIA NIM",
    description="Conversa con NVIDIA Nemotron 3 Super.",
    flagging_mode="never",
)

In [11]:
# Lanzamos el chatbot
chatbot.launch()

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://9c2cde7b44a24784bb.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [ ]:
chatbot.launch(
    server_name="0.0.0.0",
    server_port=8081,
    show_error=True
)

In [12]:
chatbot.close()

Closing server running on port: 7860


## 4. Chatbot con Streaming

El streaming muestra la respuesta a medida que se genera, token por token.
En Gradio esto se logra con `yield` en lugar de `return` — la función se convierte
en un generador que devuelve la respuesta parcial en cada paso.

In [13]:
def stream_response(message: str):
    """Genera una respuesta de NVIDIA NIM en modo streaming."""
    respuesta = ""

    stream = client.chat.completions.create(
        model=MODELO,
        messages=[
            {"role": "system", "content": "Eres un asistente útil."},
            {"role": "user", "content": message},
        ],
        temperature=1.0,
        top_p=0.95,
        stream=True,
    )

    for chunk in stream:
        texto = chunk.choices[0].delta.content if chunk.choices else None
        if texto:
            respuesta += texto
            yield respuesta

In [14]:
# Gradio detecta automáticamente que la función es un generador (usa yield)
# y activa el modo streaming en la interfaz
streaming_chatbot = gr.Interface(
    fn=stream_response,
    inputs="textbox",
    outputs="text",
    title="Chatbot con Streaming",
    description="Las respuestas aparecen en tiempo real a medida que se generan",
    flagging_mode="never"
)

In [15]:
streaming_chatbot.launch(
    server_name="0.0.0.0",
    server_port=8081,
    show_error=True
)

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://0df1a96cac16ca9ef9.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [19]:
streaming_chatbot.close()

Closing server running on port: 8081


## 5. Chatbot con memoria

`gr.ChatInterface` está diseñado para chatbots con historial. Gradio gestiona la
interfaz y entrega a nuestra función el mensaje actual y los intercambios
anteriores. Nosotros convertimos ese historial al formato estándar de mensajes
de la API.

In [20]:
def get_text(content):
    """Extrae texto de los formatos de contenido usados por Gradio."""
    if isinstance(content, str):
        return content
    if isinstance(content, list) and content:
        first = content[0]
        return first.get("text", "") if isinstance(first, dict) else str(first)
    return str(content)


def history_to_messages(history: list) -> list:
    """Convierte historiales nuevos o antiguos de Gradio a mensajes OpenAI."""
    messages = []
    for entry in history:
        if isinstance(entry, dict):
            messages.append({
                "role": entry["role"],
                "content": get_text(entry["content"]),
            })
        elif isinstance(entry, (list, tuple)) and len(entry) == 2:
            user_msg, assistant_msg = entry
            messages.append({"role": "user", "content": get_text(user_msg)})
            if assistant_msg:
                messages.append({
                    "role": "assistant",
                    "content": get_text(assistant_msg),
                })
    return messages


def chat_con_memoria(message: str, history: list):
    messages = [{"role": "system", "content": "Eres un asistente útil."}]
    messages.extend(history_to_messages(history))
    messages.append({"role": "user", "content": message})

    respuesta = ""
    stream = client.chat.completions.create(
        model=MODELO,
        messages=messages,
        temperature=1.0,
        top_p=0.95,
        stream=True,
    )
    for chunk in stream:
        texto = chunk.choices[0].delta.content if chunk.choices else None
        if texto:
            respuesta += texto
            yield respuesta

In [21]:
# gr.ChatInterface se encarga de toda la UI del chat:
# burbuja de mensajes, historial visual, campo de entrada, etc.
memory_chatbot = gr.ChatInterface(
    fn=chat_con_memoria,
    title="Chatbot con Memoria",
    description="El asistente recuerda el contexto de toda la conversacion.",
    examples=[
        "Cuentame sobre el aprendizaje automatico",
        "Como funcionan las redes neuronales?",
        "Cual es la diferencia entre IA y ML?"
    ],
    flagging_mode="never"
)

In [22]:
memory_chatbot.launch(
    server_name="0.0.0.0",
    server_port=8081,
    show_error=True
)

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://b15fc3cc09862a2d0e.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [23]:
memory_chatbot.close()

Closing server running on port: 8081


## 6. Generador de Menús de Restaurante

Veamos un ejemplo de aplicación con múltiples entradas. Gradio puede manejar
formularios complejos — aquí el usuario proporciona tres campos y obtiene
un menú generado por IA en formato Markdown.

In [ ]:
def generate_menu(nombre_restaurante: str, tipo_cocina: str, requisitos_especiales: str = "Ninguno") -> str:
    """Genera un menú de restaurante usando NVIDIA NIM."""
    prompt = f"""
    Crea un menú para "{nombre_restaurante}", un restaurante de cocina {tipo_cocina}.
    Requisitos especiales: {requisitos_especiales}

    Incluye 3 entradas, 4 platos principales y 2 postres. Para cada elemento
    incluye nombre, descripción breve y precio en pesos colombianos. Formatea
    la respuesta en Markdown y agrega una introducción corta.
    """

    response = client.chat.completions.create(
        model=MODELO,
        messages=[
            {
                "role": "system",
                "content": "Eres un consultor experto en restaurantes y diseño de menús.",
            },
            {"role": "user", "content": prompt},
        ],
        temperature=1.0,
        top_p=0.95,
    )
    return response.choices[0].message.content

In [ ]:
# Interfaz con múltiples inputs y output en Markdown
menu_generator = gr.Interface(
    fn=generate_menu,
    inputs=[
        gr.Textbox(label="Nombre del Restaurante"),
        gr.Textbox(label="Tipo de Cocina (ej. italiana, japonesa, colombiana)"),
        gr.Textbox(
            label="Requisitos Especiales (Opcional)",
            placeholder="ej. opciones vegetarianas, sin gluten"
        )
    ],
    outputs=gr.Markdown(label="Menu Generado"),
    title="Generador de Menus con IA",
    description="Crea un menu profesional para tu restaurante en segundos",
    flagging_mode="never"
)

In [ ]:
menu_generator.launch(
    server_name="0.0.0.0",
    server_port=8081,
    show_error=True
)

In [ ]:
menu_generator.close()

## 7. Personalizando el System Prompt

Una extensión útil: permitir al usuario definir el comportamiento del asistente
cambiando el system prompt desde la interfaz.

In [ ]:
def chat_personalizable(message: str, history: list, system_prompt: str):
    """Chat en el que el usuario personaliza el system prompt."""
    system_prompt = system_prompt or "Eres un asistente útil."

    messages = [{"role": "system", "content": system_prompt}]
    messages.extend(history_to_messages(history))
    messages.append({"role": "user", "content": message})

    respuesta = ""
    stream = client.chat.completions.create(
        model=MODELO,
        messages=messages,
        temperature=1.0,
        top_p=0.95,
        stream=True,
    )
    for chunk in stream:
        texto = chunk.choices[0].delta.content if chunk.choices else None
        if texto:
            respuesta += texto
            yield respuesta

In [ ]:
# Usamos gr.ChatInterface con additional_inputs para agregar el campo de system prompt
custom_chatbot = gr.ChatInterface(
    fn=chat_personalizable,
    title="Chatbot Personalizable",
    description="Define como quieres que se comporte el asistente.",
    additional_inputs=[
        gr.Textbox(
            label="System Prompt",
            placeholder="ej. Eres un experto en Python que responde solo con codigo",
            value="Eres un asistente util y conciso."
        )
    ],
    examples=[
        ["Explica que es una red neuronal", "Eres un profesor universitario."],
        ["Escribe un haiku sobre la programacion", "Eres un poeta."],
    ],
    flagging_mode="never"
)

In [ ]:
custom_chatbot.launch(
    server_name="0.0.0.0",
    server_port=8081,
    show_error=True
)

In [ ]:
custom_chatbot.close()